### Set up environment

In [17]:
%pip install accelerate bitsandbytes google-cloud-aiplatform peft trl scikit-learn tokenizers torch transformers unsloth evaluate python-dotenv sentencepiece protobuf matplotlib

Note: you may need to restart the kernel to use updated packages.


### import library

In [18]:
import os
import sys
import torch
import evaluate
import numpy as np
from trl import SFTTrainer
from tqdm.auto import tqdm
from torch.optim import AdamW
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from accelerate import Accelerator
from torch.utils.data import DataLoader
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

### Login huggingface

In [19]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face successfully.")
else:
    print("HUGFACE_TOKEN not found in environment variables. Please set it to log in.")

Logged in to Hugging Face successfully.


### Device

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load data and preproces

In [21]:
print("Loading dataset...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
print("Dataset loaded successfully.")
print("Dataset structure:")
print(dataset)

Loading dataset...
Dataset loaded successfully.
Dataset structure:
Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})


In [22]:
def format_prompt(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]
    text = []
    for instruction, context, response in zip(instruction, context, response):
        prompt = f"<|im_start|>user{instruction}\n{context}<|im_end|>\n<|im_start|>assistan\n{response}<|im_end|>"
        text.append(prompt)
    return {"text": text}
dataset = dataset.map(format_prompt, batched=True)

### load model and training

In [23]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

config = TrainingArguments(
    output_dir = "./qwen-2.5b-full-finetuned",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-5,
    num_train_epochs = 3,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_torch",
    report_to = "none",
    save_steps = 10,
    max_steps = 20
)


trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    args = config,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.023400
2,2.727900
3,2.089600
4,2.514000
5,2.532000
6,1.980800
7,2.032000
8,2.036300
9,2.173800
10,2.410900


TrainOutput(global_step=20, training_loss=2.287069147825241, metrics={'train_runtime': 685.3964, 'train_samples_per_second': 0.233, 'train_steps_per_second': 0.029, 'total_flos': 94828683755520.0, 'train_loss': 2.287069147825241, 'epoch': 0.010658140154543033})

### Save model

In [24]:
trainer.push_to_hub("qwen-2.5b-finetuned")

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]









training_args.bin: 100%|██████████| 6.22k/6.22k [00:00<00:00, 12.4kB/s]























tokenizer.json: 100%|██████████| 11.4M/11.4M [00:02<00:00, 4.47MB/s]










































































































































































































































































































































































































































































































































model.safetensors: 100%|██████████| 1.98G/1.98G [01:46<00:00, 18.6MB/s]
Upload 3 LFS files: 100%|██████████| 3/3 [01:46<00:00, 35.55s/it] 


CommitInfo(commit_url='https://huggingface.co/HalogenFlo/qwen-2.5b-full-finetuned/commit/13a5defca12d231dcc69cc8adee8feef6f97627b', commit_message='qwen-2.5b-finetuned', commit_description='', oid='13a5defca12d231dcc69cc8adee8feef6f97627b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HalogenFlo/qwen-2.5b-full-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='HalogenFlo/qwen-2.5b-full-finetuned'), pr_revision=None, pr_num=None)

### Test model

In [29]:

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = trainer.model.generate(**inputs, max_length=2048, do_sample=True, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response
test_prompt = "What is the capital of France?"
response = generate_response(test_prompt)
print("Prompt:", test_prompt)
print("Response:", response)

Prompt: What is the capital of France?
Response: What is the capital of France? The capital of France is Paris. It is also one of the most popular cities in Europe, and has a rich history dating back over 2000 years.

The city was originally founded by the Romans as "Paris", and later became known as "Paris" in French. However, it is now commonly referred to simply as "Paris". The name comes from the Latin word for "city".

The city is located on the Mediterranean coast, with its port at the mouth of the Seine River. It has an area of about 174 square kilometers (68 square miles), and has a population of approximately 3 million people. Paris is home to many famous landmarks such as Notre-Dame Cathedral, Eiffel Tower, Louvre Museum, Champs-Élysées, and many other attractions throughout the city. 

In addition to being the capital of France, Paris is also one of the most important cultural centers in the world, hosting some of the world's most renowned museums, theaters, and music venues